In [ ]:
import cv2
import time
import subprocess
import random
import numpy as np
from gtts import gTTS
import pygame
from ultralytics import YOLO

pygame.mixer.init()

model = YOLO("yolov8n.pt")

#using coco dataset here (coincidence that it's that number)
phone_class_id = 67
#15 minutes
phone_threshold_seconds = 900

insults_gesture = [
#here i just add specific/creative insults/reminders for whenever it detects 67
]

insults_phone = [
#here i just add specific/creative insults/reminders for whenever I'm on my phone for too long
]



def speak(text):
    print(f"\n[Insult Bot]: {text}\n")
    try:
        pygame.mixer.music.stop()
        pygame.mixer.music.unload()

        tts = gTTS(text=text, lang='en')
        tts.save("temp_speech.mp3")
        pygame.mixer.music.load("temp_speech.mp3")
        pygame.mixer.music.play()
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
        pygame.mixer.music.unload()
    except Exception as e:
        print(f"Audio error: {e}")



def check_calcurse_deadlines():
    try:
        result = subprocess.run(["calcurse", "-r5"], capture_output=True, text=True)
        output = result.stdout.strip()
        return len(output) > 0
    except Exception as e:
        print(f"Calendar check error: {e}")
        return False

cap = cv2.VideoCapture(0)

prev_gray = None
gesture_cooldown = 0
phone_timer_start = None



CALENDAR_CHECK_INTERVAL = 1800  # 30 minutes
PHONE_THRESHOLD_SECONDS = 900    # 15 minutes


print("Starting... Press 'q' to quit.")



try:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        current_time = time.time()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if prev_gray is not None:

            flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            v_motion = flow[..., 1]

            h, w = gray.shape
            left_half_v = v_motion[:, :w // 2]
            right_half_v = v_motion[:, w // 2:]
            avg_left = np.mean(left_half_v)
            avg_right = np.mean(right_half_v)

            is_opposite_motion = (avg_left * avg_right < -0.2) and (abs(avg_left) > 0.5 or abs(avg_right) > 0.5)

            if is_opposite_motion:
                if current_time - gesture_cooldown > 5:
                    speak(random.choice(insults_gesture))
                    gesture_cooldown = current_time


        prev_gray = gray

        results = model(frame, verbose=False)
        phone_detected = False

        for r in results:
            for box in r.boxes:
                if int(box.cls[0]) == phone_class_id:
                    phone_detected = True
                    break

        if phone_detected:
            if phone_timer_start is None:
                phone_timer_start = current_time
            elif current_time - phone_timer_start > phone_threshold_seconds:
                if check_calcurse_deadlines():
                    speak(random.choice(insults_phone))
                phone_timer_start = current_time
        else:
            phone_timer_start = None

        #camera output
        cv2.imshow('Insult Bot - Vision Feed', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

finally:
    cap.release()
    cv2.destroyAllWindows()